In [ ]:
import copy
import json
import math
import pickle
import random
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "training_data"

sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT    =", DATA_ROOT)
print("torch        =", torch.__version__)
print("cuda         =", torch.cuda.is_available())


In [ ]:
# Data and target settings
DATASET_NAME = "mean_curvature_smooth"
TARGET_INDICES = [0]
TARGET_INDEX_FOR_PLOTS = 0
USE_GLOBAL_FEATURES = True

# Filtering and preprocessing settings
MISSING_COMPLEXITY_GROUP = {
    "dataset": "20251201",
    "timepoint": "day4p5",
    "fill_value": 2.1,
}
SPHERICITY_MAX = 0.92
SPHERICAL_MARKER_DIVERSITY_MIN = 0.5
COMPLEXITY_MIN = 2.0
INTERPOLATE_TARGET_OUTLIERS = True
OUTLIER_CLIP_QUANTILES = (0.005, 0.995)

# Split and scan settings
VAL_FRAC = 0.2
SPLIT_SEED = None
FORCED_VAL_KEYS = set()
HIDDEN_DIMS = [8, 16, 32, 64, 2 * 64, 3 * 64, 4 * 64]
MODEL_SEEDS = [0, 1, 2, 3, 4]
CONSTANT_FEATURE_VALUE = 0.0

# Baseline model settings
DROPOUT = 0.1
NORM = "batch"
RESIDUAL = True
NUM_LAYERS = 0

# Training settings
LR = 3e-4
BATCH_SIZE = 128
MAX_EPOCHS = 2000
PATIENCE = 30
NUM_WORKERS = 4
PREDICT_BATCH_SIZE = 128

# Experiment output settings
EXPERIMENT_GROUP = "constant_global_depth0_baseline_scan"
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"run_{RUN_TIMESTAMP}"
SAVE_DIR = PROJECT_ROOT / "results_experiments" / EXPERIMENT_GROUP / RUN_NAME
FIGURES_DIR = SAVE_DIR / "figures"


In [ ]:
def _safe_filename(name):
    text = str(name).strip().replace("/", "_")
    chars = [ch if (ch.isalnum() or ch in "._-") else "_" for ch in text]
    cleaned = "".join(chars).strip("._-")
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned or "figure"


def ensure_figure_dir():
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    return FIGURES_DIR


def save_mpl_figure(fig, name, *, close=False):
    path = ensure_figure_dir() / f"{_safe_filename(name)}.pdf"
    fig.savefig(path, bbox_inches="tight", transparent=True)
    print(f"Saved figure -> {path}")
    if close:
        plt.close(fig)
    return path


def set_training_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
from src.data.io import load_graph_dataset_from_dir, select_graph_targets
from src.data.metadata import (
    attach_metadata_to_graphs,
    load_aux_metadata_for_dir,
    load_marker_names_from_dir,
    print_graph_and_metadata_fields,
)

data_dir = DATA_ROOT / DATASET_NAME
graphs = load_graph_dataset_from_dir(str(data_dir))
print(f"Loaded {len(graphs)} organoids.")
if graphs:
    print("Raw y shape:", tuple(graphs[0].y.shape))

meta = load_aux_metadata_for_dir(str(data_dir))
attached = attach_metadata_to_graphs(graphs, meta, exclude_keys=None)
print(f"Attached metadata to {attached}/{len(graphs)} graphs.")

graphs = select_graph_targets(graphs, target_indices=TARGET_INDICES, inplace=False)
if graphs:
    print("Selected y shape:", tuple(graphs[0].y.shape))

marker_names = load_marker_names_from_dir(str(data_dir))
if marker_names is None:
    n_markers = int(graphs[0].x.size(1)) if graphs else 0
    marker_names = [f"marker_{i}" for i in range(n_markers)]
print(f"Loaded {len(marker_names)} markers:", marker_names)

print_graph_and_metadata_fields(graphs)


In [ ]:
# Optional categorical metadata filters can be added here.


In [ ]:
from src.data.filters import (
    filter_graphs_by_marker_diversity,
    filter_graphs_by_numeric_metadata,
    filter_graphs_by_sphericity,
)
from src.data.metadata import fill_missing_metadata_for_group
from src.data.preprocessing import interpolate_target_outliers_from_neighbors

graphs = fill_missing_metadata_for_group(
    graphs,
    field="complexity",
    fill_value=MISSING_COMPLEXITY_GROUP["fill_value"],
    dataset=MISSING_COMPLEXITY_GROUP["dataset"],
    timepoint=MISSING_COMPLEXITY_GROUP["timepoint"],
)

graphs, g_spherical = filter_graphs_by_sphericity(
    graphs,
    max_sphericity=SPHERICITY_MAX,
    print_summary=True,
    return_rejected=True,
)

g_spherical = filter_graphs_by_marker_diversity(
    g_spherical,
    min_score=SPHERICAL_MARKER_DIVERSITY_MIN,
    print_summary=True,
)

graphs = filter_graphs_by_numeric_metadata(
    graphs,
    key="complexity",
    min_value=COMPLEXITY_MIN,
    allow_missing=False,
    inplace=False,
    print_summary=True,
)

# graphs = graphs + g_spherical

print(f"After filtering and spherical rescue: {len(graphs)} organoids.")

if INTERPOLATE_TARGET_OUTLIERS:
    graphs, outlier_info = interpolate_target_outliers_from_neighbors(
        graphs,
        target_indices=None,
        clip_quantiles=OUTLIER_CLIP_QUANTILES,
    )
else:
    outlier_info = None


In [ ]:
from src.data.metadata import add_log_metadata_features, promote_metadata_to_graph_tensors

field_specs = [
    {
        "meta_keys": [
            "log_surface_area",
            "log_volume",
            "log_volume_over_area",
            "log_num_cells",
        ],
        "attr_name": "global_feat",
        "kind": "graph_vector",
        "dtype": torch.float32,
    },
]

if USE_GLOBAL_FEATURES:
    graphs = add_log_metadata_features(graphs, inplace=False)
    graphs = promote_metadata_to_graph_tensors(graphs, field_specs, inplace=False)
    print("Promoted metadata fields to graph tensor attributes.")
else:
    print("Global features disabled; no global_feat attribute was attached.")


In [ ]:
from src.data.metadata import infer_global_dim, snapshot_graph_metadata, strip_graph_metadata
from src.data.splits import graph_metadata_key, train_val_split_graphs
from src.data.target_transforms import AsinhStandardizeTransform, standardize_graph_global_features

g_train, g_val, split_info = train_val_split_graphs(
    graphs,
    val_frac=VAL_FRAC,
    seed=SPLIT_SEED,
    force_val_keys=FORCED_VAL_KEYS,
    key_fn=graph_metadata_key,
)
print(f"Split -> train: {len(g_train)} | val: {len(g_val)}")

val_meta_lookup = snapshot_graph_metadata(g_val)

g_train = strip_graph_metadata(g_train, inplace=False)
g_val = strip_graph_metadata(g_val, inplace=False)

target_transform = AsinhStandardizeTransform(robust=True).fit(g_train)
target_transform.transform_graphs(g_train)
target_transform.transform_graphs(g_val)

center_global, scale_global = None, None
if USE_GLOBAL_FEATURES:
    center_global, scale_global = standardize_graph_global_features(
        g_train,
        g_val,
        attr_name="global_feat",
        robust=False,
    )

global_dim = infer_global_dim(g_train)
print("global_dim =", global_dim)
print("target_transform =", target_transform.name)


In [ ]:
def snapshot_marker_features(graphs_in):
    return [g.x.detach().cpu().clone() for g in graphs_in]


def replace_marker_features_with_constant(graphs_in, *, constant_value=0.0):
    graphs_out = [copy.deepcopy(g) for g in graphs_in]
    for g in graphs_out:
        n_nodes = int(g.x.shape[0])
        g.x = torch.full(
            (n_nodes, 1),
            float(constant_value),
            dtype=g.x.dtype,
            device=g.x.device,
        )
    return graphs_out


reference_marker_x_train = snapshot_marker_features(g_train)
reference_marker_x_val = snapshot_marker_features(g_val)
reference_marker_names = list(marker_names)

g_train_constant = replace_marker_features_with_constant(
    g_train,
    constant_value=CONSTANT_FEATURE_VALUE,
)
g_val_constant = replace_marker_features_with_constant(
    g_val,
    constant_value=CONSTANT_FEATURE_VALUE,
)

print(f"Replaced marker features with one constant node feature: {CONSTANT_FEATURE_VALUE:g}")
print("Original markers retained for evaluation groups:", reference_marker_names)
print("Training x shape:", tuple(g_train_constant[0].x.shape))
print("Validation x shape:", tuple(g_val_constant[0].x.shape))


In [ ]:
from src.training.loop import TrainConfig, train

cfg = TrainConfig(
    loss_name="gaussian",
    lr=LR,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
)

device = cfg.device if "cfg" in globals() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device =", device)


In [ ]:
from src.models.gnn import GINCurvature
from src.data.metadata import infer_global_dim

hidden_dims = list(HIDDEN_DIMS)
model_seeds = list(MODEL_SEEDS)
trained_models = []
training_logs = []

for hidden_dim in hidden_dims:
    for seed in model_seeds:
        print()
        print("=" * 80)
        print(f"Training constant+globals depth-0 baseline | hidden_dim={hidden_dim} | seed={seed}")
        set_training_seed(seed)

        model = GINCurvature(
            n_markers=int(g_train_constant[0].x.size(1)),
            global_dim=infer_global_dim(g_train_constant),
            hidden_dim=int(hidden_dim),
            num_layers=int(NUM_LAYERS),
            dropout=DROPOUT,
            residual=RESIDUAL,
            norm=NORM,
            train_eps=True,
        )
        model, metrics, history = train(model, g_train_constant, g_val_constant, cfg)
        trained_models.append({
            "hidden_dim": int(hidden_dim),
            "seed": int(seed),
            "model": model,
        })
        training_logs.append({
            "hidden_dim": int(hidden_dim),
            "seed": int(seed),
            "metrics": metrics,
            "history": history,
        })

print(f"Finished training {len(trained_models)} baseline models.")


In [ ]:
from src.inference.predict import predict_targets
from src.analysis.marker_stats import compute_markerwise_nll, compute_markerwise_residuals
from src.analysis.prediction_analysis import (
    _aggregate_subset_metrics,
    compute_global_uncertainty_correlation,
    compute_markerwise_uncertainty_correlation,
)


def _select_target_column(values, target_index=0):
    arr = np.asarray(values, dtype=np.float64)
    if arr.ndim == 2:
        return arr[:, int(target_index)]
    return arr.reshape(-1)


def _concatenate_reference_markers(reference_x):
    arrays = []
    for x in reference_x:
        arr = x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)
        arrays.append(arr)
    return np.concatenate(arrays, axis=0).astype(np.float64)


def eval_model_per_reference_marker(
    model,
    eval_graphs,
    reference_x,
    *,
    device,
    target_transform=None,
    eps=1e-12,
    center_only=False,
    target_index=0,
    batch_size=128,
):
    y, mu, log_var, _ = predict_targets(
        eval_graphs,
        model,
        device=device,
        batch_size=batch_size,
        return_log_var=True,
        center_only=center_only,
        target_transform=target_transform,
    )

    y_eval = _select_target_column(y, target_index=target_index)
    mu_eval = _select_target_column(mu, target_index=target_index)
    log_var_eval = _select_target_column(log_var, target_index=target_index)
    var = np.exp(log_var_eval)
    X_ref = _concatenate_reference_markers(reference_x)
    if X_ref.shape[0] != y_eval.shape[0]:
        raise ValueError(f"Reference marker rows {X_ref.shape[0]} do not match predictions {y_eval.shape[0]}.")

    residuals_model, residuals_base, n_pos_resid, mu_pos_resid = compute_markerwise_residuals(
        y_eval,
        mu_eval,
        X_ref,
    )
    mse_model = np.array([np.mean(r**2) if r.size > 0 else np.nan for r in residuals_model])
    mse_base = np.array([np.mean(r**2) if r.size > 0 else np.nan for r in residuals_base])

    sem_mse_model = np.array([
        np.std(r**2, ddof=1) / np.sqrt(r.size) if r.size > 1 else np.nan
        for r in residuals_model
    ])
    sem_mse_base = np.array([
        np.std(r**2, ddof=1) / np.sqrt(r.size) if r.size > 1 else np.nan
        for r in residuals_base
    ])

    Xb = X_ref > 0.5
    var_model = np.array([
        np.mean(var[Xb[:, m]]) if np.any(Xb[:, m]) else np.nan
        for m in range(X_ref.shape[1])
    ])
    sem_var_model = np.array([
        np.std(var[Xb[:, m]], ddof=1) / np.sqrt(np.sum(Xb[:, m])) if np.sum(Xb[:, m]) > 1 else np.nan
        for m in range(X_ref.shape[1])
    ])

    var_base = mse_base
    sem_var_base = sem_mse_base

    nll_model_list, nll_base_list, delta_mean_list, delta_var_list, n_pos_nll = compute_markerwise_nll(
        y_true=y_eval,
        mu_model=mu_eval,
        log_var_model=log_var_eval,
        X=X_ref,
        eps=eps,
    )

    nll_model = np.array([np.mean(v) if v.size > 0 else np.nan for v in nll_model_list])
    nll_base = np.array([np.mean(v) if v.size > 0 else np.nan for v in nll_base_list])
    sem_nll_model = np.array([
        np.std(v, ddof=1) / np.sqrt(v.size) if v.size > 1 else np.nan
        for v in nll_model_list
    ])
    sem_nll_base = np.array([
        np.std(v, ddof=1) / np.sqrt(v.size) if v.size > 1 else np.nan
        for v in nll_base_list
    ])

    rho_marker, pval_marker, n_pos_corr = compute_markerwise_uncertainty_correlation(
        y=y_eval,
        mu=mu_eval,
        log_var=log_var_eval,
        X=X_ref,
    )
    rho_global, pval_global = compute_global_uncertainty_correlation(
        y=y_eval,
        mu=mu_eval,
        log_var=log_var_eval,
    )

    row_pos = (X_ref > 0.5).sum(axis=1)
    aggregate = {
        "any_marker": _aggregate_subset_metrics(y_eval, mu_eval, log_var_eval, X_ref, row_pos > 0, eps=eps),
        "no_marker": _aggregate_subset_metrics(y_eval, mu_eval, log_var_eval, X_ref, row_pos == 0, eps=eps),
        "all_nodes": _aggregate_subset_metrics(y_eval, mu_eval, log_var_eval, X_ref, np.ones_like(row_pos, dtype=bool), eps=eps),
    }

    return {
        "mse_model": mse_model,
        "sem_mse_model": sem_mse_model,
        "mse_base": mse_base,
        "sem_mse_base": sem_mse_base,
        "var_model": var_model,
        "sem_var_model": sem_var_model,
        "var_base": var_base,
        "sem_var_base": sem_var_base,
        "nll_model": nll_model,
        "sem_nll_model": sem_nll_model,
        "nll_base": nll_base,
        "sem_nll_base": sem_nll_base,
        "rho_marker": rho_marker,
        "pval_marker": pval_marker,
        "n_pos_corr": n_pos_corr,
        "rho_global": rho_global,
        "pval_global": pval_global,
        "aggregate": aggregate,
    }


In [ ]:
eval_rows = []
marker_rows = []
eval_outputs = []

for pack in trained_models:
    hidden_dim = int(pack["hidden_dim"])
    seed = int(pack["seed"])
    model = pack["model"]
    model.eval()

    out = eval_model_per_reference_marker(
        model,
        g_val_constant,
        reference_marker_x_val,
        device=device,
        target_transform=target_transform,
        target_index=TARGET_INDEX_FOR_PLOTS,
        batch_size=PREDICT_BATCH_SIZE,
    )
    eval_outputs.append({
        "hidden_dim": hidden_dim,
        "seed": seed,
        "out": out,
    })

    for aggregate_name, aggregate in out["aggregate"].items():
        row = {
            "hidden_dim": hidden_dim,
            "seed": seed,
            "subset": aggregate_name,
        }
        row.update({k: aggregate[k] for k in [
            "n",
            "mse_model", "sem_mse_model",
            "mse_base", "sem_mse_base",
            "var_model", "sem_var_model",
            "var_base", "sem_var_base",
            "nll_model", "sem_nll_model",
            "nll_base", "sem_nll_base",
            "rho", "pval",
        ]})
        eval_rows.append(row)

    for mi, marker_name in enumerate(reference_marker_names):
        marker_rows.append({
            "hidden_dim": hidden_dim,
            "seed": seed,
            "marker_index": mi,
            "marker_name": marker_name,
            "mse_model": out["mse_model"][mi],
            "sem_mse_model": out["sem_mse_model"][mi],
            "mse_base": out["mse_base"][mi],
            "sem_mse_base": out["sem_mse_base"][mi],
            "nll_model": out["nll_model"][mi],
            "sem_nll_model": out["sem_nll_model"][mi],
            "nll_base": out["nll_base"][mi],
            "sem_nll_base": out["sem_nll_base"][mi],
            "rho_marker": out["rho_marker"][mi],
            "pval_marker": out["pval_marker"][mi],
            "n_pos_corr": out["n_pos_corr"][mi],
        })

eval_df = pd.DataFrame(eval_rows)
marker_eval_df = pd.DataFrame(marker_rows)

stability_df = (
    eval_df
    .groupby(["subset", "hidden_dim"], as_index=False)
    .agg(
        n_runs=("seed", "nunique"),
        mse_mean=("mse_model", "mean"),
        mse_std=("mse_model", "std"),
        nll_mean=("nll_model", "mean"),
        nll_std=("nll_model", "std"),
        rho_mean=("rho", "mean"),
        rho_std=("rho", "std"),
    )
)

marker_stability_df = (
    marker_eval_df
    .groupby(["marker_name", "marker_index", "hidden_dim"], as_index=False)
    .agg(
        n_runs=("seed", "nunique"),
        mse_mean=("mse_model", "mean"),
        mse_std=("mse_model", "std"),
        nll_mean=("nll_model", "mean"),
        nll_std=("nll_model", "std"),
        rho_mean=("rho_marker", "mean"),
        rho_std=("rho_marker", "std"),
    )
)

stability_df.head()


In [ ]:
plt.rcParams.update({
    "axes.spines.top": True,
    "axes.spines.right": True,
    "axes.grid": True,
    "grid.alpha": 0.25,
})


def _plot_aggregate_metric(metric_mean, metric_std, ylabel, title, filename):
    subset_order = ["all_nodes", "any_marker", "no_marker"]
    subset_labels = {
        "all_nodes": "all nodes",
        "any_marker": "any marker+",
        "no_marker": "no marker+",
    }
    colors = {
        "all_nodes": "#003961",
        "any_marker": "#6e0000",
        "no_marker": "#2ca02c",
    }

    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    for subset in subset_order:
        table = stability_df[stability_df["subset"] == subset].sort_values("hidden_dim")
        if len(table) == 0:
            continue
        x = table["hidden_dim"].to_numpy(dtype=float)
        y = table[metric_mean].to_numpy(dtype=float)
        s = table[metric_std].fillna(0.0).to_numpy(dtype=float)
        ax.plot(x, y, "-o", color=colors[subset], linewidth=2, markersize=5, label=subset_labels[subset])
        ax.fill_between(x, y - s, y + s, color=colors[subset], alpha=0.16)

        raw = eval_df[eval_df["subset"] == subset]
        for seed, raw_seed in raw.groupby("seed"):
            raw_seed = raw_seed.sort_values("hidden_dim")
            raw_metric = metric_mean.replace("_mean", "_model") if metric_mean != "rho_mean" else "rho"
            ax.plot(
                raw_seed["hidden_dim"],
                raw_seed[raw_metric],
                color=colors[subset],
                alpha=0.18,
                linewidth=0.9,
            )

    ax.set_xscale("log")
    ax.set_xticks(hidden_dims)
    ax.set_xticklabels([str(v) for v in hidden_dims])
    ax.set_xlabel("hidden dimension")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)
    plt.tight_layout()
    save_mpl_figure(fig, filename)
    plt.show()
    return fig, ax


fig_agg_mse, ax_agg_mse = _plot_aggregate_metric(
    "mse_mean",
    "mse_std",
    "Mean Squared Error",
    "Constant+globals depth-0 baseline MSE stability",
    "aggregate_mse_by_hidden_dim_constant_global_depth0",
)

fig_agg_nll, ax_agg_nll = _plot_aggregate_metric(
    "nll_mean",
    "nll_std",
    "Negative Log-Likelihood",
    "Constant+globals depth-0 baseline NLL stability",
    "aggregate_nll_by_hidden_dim_constant_global_depth0",
)

fig_agg_rho, ax_agg_rho = _plot_aggregate_metric(
    "rho_mean",
    "rho_std",
    "Spearman corr(var, residual²)",
    "Constant+globals depth-0 uncertainty-error stability",
    "aggregate_uncertainty_error_correlation_by_hidden_dim_constant_global_depth0",
)


In [ ]:
def _plot_marker_metric(metric_mean, metric_std, ylabel, title, filename, *, baseline_hline=None):
    n_markers = len(reference_marker_names)
    n_cols = 4
    n_rows = int(math.ceil(n_markers / n_cols))
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.3 * n_cols, 2.8 * n_rows),
        sharex=True,
        squeeze=False,
    )
    axes = axes.ravel()
    color = "#003961"

    for mi, marker_name in enumerate(reference_marker_names):
        ax = axes[mi]
        table = marker_stability_df[marker_stability_df["marker_name"] == marker_name].sort_values("hidden_dim")
        if len(table) == 0:
            ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", va="center")
            continue
        x = table["hidden_dim"].to_numpy(dtype=float)
        y = table[metric_mean].to_numpy(dtype=float)
        s = table[metric_std].fillna(0.0).to_numpy(dtype=float)
        ax.plot(x, y, "-o", color=color, linewidth=1.8, markersize=4)
        ax.fill_between(x, y - s, y + s, color=color, alpha=0.16)
        if baseline_hline is not None:
            ax.axhline(baseline_hline, color="0.4", linestyle="--", linewidth=1.0)
        ax.set_title(marker_name, fontsize=10)
        ax.set_xscale("log")
        ax.set_xticks(hidden_dims)
        ax.set_xticklabels([str(v) for v in hidden_dims], rotation=45, ha="right")
        ax.set_xlabel("hidden dim")
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.25)

    for ax in axes[n_markers:]:
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    save_mpl_figure(fig, filename)
    plt.show()
    return fig, axes


fig_marker_mse, axes_marker_mse = _plot_marker_metric(
    "mse_mean",
    "mse_std",
    "MSE",
    "Marker-wise MSE stability | constant+globals depth-0 baseline",
    "markerwise_mse_by_hidden_dim_constant_global_depth0",
)

fig_marker_nll, axes_marker_nll = _plot_marker_metric(
    "nll_mean",
    "nll_std",
    "NLL",
    "Marker-wise NLL stability | constant+globals depth-0 baseline",
    "markerwise_nll_by_hidden_dim_constant_global_depth0",
)

fig_marker_rho, axes_marker_rho = _plot_marker_metric(
    "rho_mean",
    "rho_std",
    "Spearman corr",
    "Marker-wise uncertainty-error correlation stability | constant+globals depth-0 baseline",
    "markerwise_uncertainty_error_correlation_by_hidden_dim_constant_global_depth0",
    baseline_hline=0.0,
)


In [ ]:
def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, np.dtype):
        return str(obj)
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def save_joint_experiment(save_dir, config, results_by_experiment, notes=None):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    with open(save_dir / "config.json", "w") as f:
        json.dump(_jsonable(config), f, indent=2)

    with open(save_dir / "results.pkl", "wb") as f:
        pickle.dump(results_by_experiment, f)

    meta = {
        "timestamp": datetime.now().isoformat(),
        "notes": notes,
        "experiments": list(results_by_experiment.keys()),
    }
    with open(save_dir / "meta.json", "w") as f:
        json.dump(_jsonable(meta), f, indent=2)

    print(f"Saved joint experiment -> {save_dir}")


In [ ]:
joint_config = {
    "experiment_group": EXPERIMENT_GROUP,
    "dataset_name": DATASET_NAME,
    "target_indices": TARGET_INDICES,
    "target_index_for_plots": TARGET_INDEX_FOR_PLOTS,
    "use_global_features": USE_GLOBAL_FEATURES,
    "filters": {
        "missing_complexity_group": MISSING_COMPLEXITY_GROUP,
        "sphericity_max": SPHERICITY_MAX,
        "spherical_marker_diversity_min": SPHERICAL_MARKER_DIVERSITY_MIN,
        "complexity_min": COMPLEXITY_MIN,
        "interpolate_target_outliers": INTERPOLATE_TARGET_OUTLIERS,
        "outlier_clip_quantiles": OUTLIER_CLIP_QUANTILES,
    },
    "split": {
        "val_frac": VAL_FRAC,
        "split_seed": SPLIT_SEED,
        "forced_val_keys": sorted(FORCED_VAL_KEYS),
    },
    "constant_global_depth0_baseline": {
        "constant_feature_value": CONSTANT_FEATURE_VALUE,
        "num_layers": NUM_LAYERS,
        "hidden_dims": hidden_dims,
        "model_seeds": model_seeds,
        "evaluation_marker_names": list(reference_marker_names),
    },
    "model_settings": {
        "dropout": DROPOUT,
        "norm": NORM,
        "residual": RESIDUAL,
    },
    "train_config": {
        "loss_name": cfg.loss_name,
        "lr": cfg.lr,
        "batch_size": cfg.batch_size,
        "max_epochs": cfg.max_epochs,
        "patience": cfg.patience,
        "num_workers": cfg.num_workers,
    },
    "field_specs": field_specs,
    "n_train_graphs": len(g_train_constant),
    "n_val_graphs": len(g_val_constant),
    "marker_names": list(reference_marker_names),
    "figures_dir": str(FIGURES_DIR),
}

results_by_experiment = {
    "eval_df": eval_df,
    "marker_eval_df": marker_eval_df,
    "stability_df": stability_df,
    "marker_stability_df": marker_stability_df,
    "training_logs": training_logs,
}

save_joint_experiment(
    save_dir=SAVE_DIR,
    config=joint_config,
    results_by_experiment=results_by_experiment,
    notes="Hidden-dimension and random-seed stability scan for the constant+globals depth-0 GIN baseline.",
)
